# Human vs LLM Dataset Comparison
Side-by-side qualitative comparison of the two extraction datasets for the 7 papers in common.

In [4]:
import pandas as pd
import json
from IPython.display import display, HTML

human = pd.read_csv('human/human_generated.csv')
llm   = pd.read_excel('llm/LLM_generated.xlsx')

COMMON_PAPERS = sorted(set(human['Filename']) & set(llm['custom_id']))
human_7 = human[human['Filename'].isin(COMMON_PAPERS)].copy()
llm_7   = llm[llm['custom_id'].isin(COMMON_PAPERS)].copy()

print(f'Human rows (7 papers): {len(human_7)}')
print(f'LLM rows   (7 papers): {len(llm_7)}')
print(f'Papers: {COMMON_PAPERS}')

Human rows (7 papers): 33
LLM rows   (7 papers): 33
Papers: ['10.1007_s10640-025-00970-6', '10.1007_s10645-008-9094-1', '10.1016_j.evolhumbehav.2006.06.001', '10.1016_j.jpubeco.2015.12.012', '10.1111_apce.12343', '10.1177_0146167216684134', '10.3390_g14050065']


## 1. Column overview — what exists in each dataset

In [5]:
# COLUMN_MAP: human col name → LLM col name
COLUMN_MAP = {
    'Empirical':                    'METHOD_empirical',
    'Controled_Or_Observational':   'METHOD_lab',
    'Lab_Or_Field':                 'METHOD_experiment',
    'CONFIG_playerCount':           'CONFIG_playerCount',
    'CONFIG_numRounds':             'CONFIG_numRounds',
    'CONFIG_showNRounds':           'CONFIG_showNRounds',
    'CONFIG_allOrNothing':          'CONFIG_allOrNothing',
    'CONFIG_defaultContribProp':    'CONFIG_defaultContribProp',
    'CONFIG_MPCR':                  'CONFIG_MPCR',
    'CONFIG_chat':                  'CONFIG_chat',
    'CONFIG_showOtherSummaries':    'CONFIG_showOtherSummaries',
    'CONFIG_showPunishmentId':      'CONFIG_showPunishmentId',
    'CONFIG_punishmentExists':      'CONFIG_punishmentExists',
    'CONFIG_punishmentCost':        'CONFIG_punishmentCost',
    'CONFIG_punishmentTech':        'CONFIG_punishmentTech',
    'CONFIG_showRewardId':          'CONFIG_showRewardId',
    'CONFIG_rewardExists':          'CONFIG_rewardExists',
    'CONFIG_rewardCost':            'CONFIG_rewardCost',
    'CONFIG_rewardTech':            'CONFIG_rewardTech',
    'DV_efficiencyReported':        'DV_efficiencyReported',
    'Dependent Variables':          'DVs',
    'Dependent Variables Definitions': 'DVs_Definitions',
}

rows = []
for h_col, l_col in COLUMN_MAP.items():
    rows.append({
        'Human column': h_col,
        'LLM column': l_col,
        'In human CSV': h_col in human.columns,
        'In LLM xlsx': l_col in llm.columns,
    })
display(pd.DataFrame(rows))

,Human column,LLM column,In human CSV,In LLM xlsx
0,Empirical,METHOD_empirical,True,True
1,Controled_Or_Observational,METHOD_lab,True,True
2,Lab_Or_Field,METHOD_experiment,True,True
3,CONFIG_playerCount,CONFIG_playerCount,True,True
4,CONFIG_numRounds,CONFIG_numRounds,True,True
5,CONFIG_showNRounds,CONFIG_showNRounds,True,True
6,CONFIG_allOrNothing,CONFIG_allOrNothing,True,True
7,CONFIG_defaultContribProp,CONFIG_defaultContribProp,True,True
8,CONFIG_MPCR,CONFIG_MPCR,True,True
9,CONFIG_chat,CONFIG_chat,True,True


## 2. DVs — naming convention mismatch (root cause of 0%)

Human uses full English descriptions; LLM uses short snake_case identifiers.
Jaccard similarity between these string sets is 0 → 0% accuracy.

In [7]:
def parse_json_safe(val):
    if pd.isna(val):
        return None
    try:
        return json.loads(str(val))
    except Exception:
        return str(val)

for paper in COMMON_PAPERS:
    h_rows = human_7[human_7['Filename'] == paper]
    l_rows = llm_7[llm_7['custom_id'] == paper]

    h_dvs = parse_json_safe(h_rows['Dependent Variables'].iloc[0])
    l_dvs = parse_json_safe(l_rows['DVs'].iloc[0])

    print('\n' + '=' * 70)
    print(f'Paper: {paper}')
    print(f'  Human DVs  : {h_dvs}')
    print(f'  LLM DVs    : {l_dvs}')

    if isinstance(h_dvs, list) and isinstance(l_dvs, list):
        h_set = {x.lower().strip() for x in h_dvs}
        l_set = {x.lower().strip() for x in l_dvs}
        intersection = h_set & l_set
        union = h_set | l_set
        jaccard = len(intersection) / len(union) if union else 0
        print(f'  Jaccard    : {jaccard:.2f}  (overlap: {intersection})')

SyntaxError: f-string: expecting '}' (2708536907.py, line 16)

## 3. DVs_Definitions — side by side per paper

In [ ]:
for paper in COMMON_PAPERS:
    h_rows = human_7[human_7['Filename'] == paper]
    l_rows = llm_7[llm_7['custom_id'] == paper]

    h_def = parse_json_safe(h_rows['Dependent Variables Definitions'].iloc[0])
    l_def = parse_json_safe(l_rows['DVs_Definitions'].iloc[0])

    print('\n' + '=' * 70)
    print(f'Paper: {paper}')
    print('  Human keys :', list(h_def.keys()) if isinstance(h_def, dict) else h_def)
    print('  LLM keys   :', list(l_def.keys()) if isinstance(l_def, dict) else l_def)


Paper: 10.1007_s10640-025-00970-6
  Human keys : ['individual/group contribution', 'change in contribution']
  LLM keys   : ['contribution', 'punishment', 'prosocial_punishment', 'antisocial_punishment', 'earnings', 'change_in_contribution']

Paper: 10.1007_s10645-008-9094-1
  Human keys : ['individual contribution to the public good', 'group average contribution']
  LLM keys   : ['contribution_rate', 'earnings', 'punishment_sent', 'punishment_received', 'contribution_change']

Paper: 10.1016_j.evolhumbehav.2006.06.001
  Human keys : ['money spent on punishment by third party (US$0–7)']
  LLM keys   : ['punishment_expenditure', 'punishment_rate', 'earnings']

Paper: 10.1016_j.jpubeco.2015.12.012
  Human keys : ['contribution to the public good', 'punishment points assigned', 'net earnings', 'self-reported happiness', 'self-reported anger']
  LLM keys   : ['contribution', 'punishment_assigned', 'earnings', 'self_reported_emotions']

Paper: 10.1111_apce.12343
  Human keys : ["subject's 

## 4. Side-by-side comparison for all mapped fields (per paper × condition)

In [ ]:
SCALAR_COLS = {k: v for k, v in COLUMN_MAP.items()
               if k not in ('Dependent Variables', 'Dependent Variables Definitions')}

records = []
for paper in COMMON_PAPERS:
    h_rows = human_7[human_7['Filename'] == paper].reset_index(drop=True)
    l_rows = llm_7[llm_7['custom_id'] == paper].reset_index(drop=True)
    n = min(len(h_rows), len(l_rows))
    for i in range(n):
        for h_col, l_col in SCALAR_COLS.items():
            h_val = h_rows.iloc[i].get(h_col, None)
            l_val = l_rows.iloc[i].get(l_col, None)
            records.append({
                'paper': paper,
                'row': i,
                'condition (human)': h_rows.iloc[i].get('Granularity', ''),
                'condition (LLM)': l_rows.iloc[i].get('data_id', ''),
                'field': h_col,
                'human': h_val,
                'llm': l_val,
            })

df_cmp = pd.DataFrame(records)
print(f'Total field comparisons: {len(df_cmp)}')
df_cmp.head(20)

Total field comparisons: 660


,paper,row,condition (human),condition (LLM),field,human,llm
0,10.1007_s10640-025-00970-6,0,Condition 0,"NoR – Part NoP (No Representation, No Punishment)",Empirical,1.0,True
1,10.1007_s10640-025-00970-6,0,Condition 0,"NoR – Part NoP (No Representation, No Punishment)",Controled_Or_Observational,1.0,True
2,10.1007_s10640-025-00970-6,0,Condition 0,"NoR – Part NoP (No Representation, No Punishment)",Lab_Or_Field,0.0,True
3,10.1007_s10640-025-00970-6,0,Condition 0,"NoR – Part NoP (No Representation, No Punishment)",CONFIG_playerCount,4.0,4
4,10.1007_s10640-025-00970-6,0,Condition 0,"NoR – Part NoP (No Representation, No Punishment)",CONFIG_numRounds,10.0,10
5,10.1007_s10640-025-00970-6,0,Condition 0,"NoR – Part NoP (No Representation, No Punishment)",CONFIG_showNRounds,1,1.0
6,10.1007_s10640-025-00970-6,0,Condition 0,"NoR – Part NoP (No Representation, No Punishment)",CONFIG_allOrNothing,1.0,1
7,10.1007_s10640-025-00970-6,0,Condition 0,"NoR – Part NoP (No Representation, No Punishment)",CONFIG_defaultContribProp,0,0.0
8,10.1007_s10640-025-00970-6,0,Condition 0,"NoR – Part NoP (No Representation, No Punishment)",CONFIG_MPCR,0.4,0.4
9,10.1007_s10640-025-00970-6,0,Condition 0,"NoR – Part NoP (No Representation, No Punishment)",CONFIG_chat,0.0,NaN


## 5. Focus on fields that show 0% in the evaluator

In [ ]:
ZERO_PCT_FIELDS = ['Empirical', 'Controled_Or_Observational', 'Lab_Or_Field',
                   'CONFIG_rewardCost', 'CONFIG_rewardTech']

zero_df = df_cmp[df_cmp['field'].isin(ZERO_PCT_FIELDS)]
display(zero_df[['paper', 'condition (human)', 'condition (LLM)', 'field', 'human', 'llm']]
        .sort_values(['field', 'paper'])
        .reset_index(drop=True))

,paper,condition (human),condition (LLM),field,human,llm
0,10.1007_s10640-025-00970-6,Condition 0,"NoR – Part NoP (No Representation, No Punishment)",CONFIG_rewardCost,NaN,NaN
1,10.1007_s10640-025-00970-6,Condition 1,"NoR – Part P (No Representation, Punishment)",CONFIG_rewardCost,NaN,NaN
2,10.1007_s10640-025-00970-6,Condition 2,"RnoM – Part NoP (Representation, No Messages, ...",CONFIG_rewardCost,NaN,NaN
3,10.1007_s10640-025-00970-6,Condition 3,"RnoM – Part P (Representation, No Messages, Pu...",CONFIG_rewardCost,NaN,NaN
4,10.1007_s10640-025-00970-6,Condition 4,"RM – Part NoP (Representation, Messages, No Pu...",CONFIG_rewardCost,NaN,NaN
...,...,...,...,...,...,...
160,10.1177_0146167216684134,Condition 1,Study 2 – Control condition,Lab_Or_Field,0.0,True
161,10.3390_g14050065,Condition 0,Sim-Sym,Lab_Or_Field,0.0,True
162,10.3390_g14050065,Condition 1,Sim-Asym,Lab_Or_Field,0.0,True
163,10.3390_g14050065,Condition 2,Seq-Sym,Lab_Or_Field,0.0,True


## 6. Per-paper, per-condition full comparison table (interactive)

In [ ]:
# Change paper_id to inspect a specific paper
paper_id = COMMON_PAPERS[0]

paper_df = df_cmp[df_cmp['paper'] == paper_id].pivot_table(
    index='field', columns='row', values=['human', 'llm'], aggfunc='first'
)
print(f'Paper: {paper_id}')
display(paper_df)

Paper: 10.1007_s10640-025-00970-6


human                                   llm        \
row                             0     1      2     3      4     5     0     1   
field                                                                           
CONFIG_MPCR                   0.4   0.4    0.4   0.4    0.4   0.4   0.4   0.4   
CONFIG_allOrNothing           1.0   1.0    1.0   1.0    1.0   1.0     1     1   
CONFIG_chat                   0.0   0.0    0.0   0.0    1.0   1.0  None  None   
CONFIG_defaultContribProp       0     0      0     0      0     0   0.0   0.0   
CONFIG_numRounds             10.0  10.0   10.0  10.0   10.0  10.0    10    10   
CONFIG_playerCount            4.0   4.0   12.0  12.0   12.0  12.0     4     4   
CONFIG_punishmentCost         NaN     1    NaN     1    NaN     1   NaN   1.0   
CONFIG_punishmentExists     FALSE  TRUE  FALSE  TRUE  FALSE  TRUE  None   1.0   
CONFIG_punishmentTech         NaN     3    NaN     3    NaN     3   NaN   3.0   
CONFIG_rewardExists           0.0   0.0    0.0   0.0    0.0   0.0  None  None   
CONFIG_showNRounds              1     1      1     1      1     1   1.0   1.0   
CONFIG_showOtherSummaries     1.0   1.0    1.0   1.0    1.0   1.0   1.0   1.0   
CONFIG_showPunishmentId       NaN   1.0    NaN   1.0    NaN   1.0   NaN  None   
Controled_Or_Observational    1.0   1.0    1.0   1.0    1.0   1.0  True  True   
DV_efficiencyReported         0.0   0.0    0.0   0.0    0.0   0.0     1     1   
Empirical                     1.0   1.0    1.0   1.0    1.0   1.0  True  True   
Lab_Or_Field                  0.0   0.0    0.0   0.0    0.0   0.0  True  True   

                                                    
row                            2     3     4     5  
field                                               
CONFIG_MPCR                  0.4   0.4   0.4   0.4  
CONFIG_allOrNothing            1     1     1     1  
CONFIG_chat                 None  None  None  None  
CONFIG_defaultContribProp    0.0   0.0   0.0   0.0  
CONFIG_numRounds              10    10    10    10  
CONFIG_playerCount             4     4     4     4  
CONFIG_punishmentCost        NaN   1.0   NaN   1.0  
CONFIG_punishmentExists     None   1.0  None   1.0  
CONFIG_punishmentTech        NaN   3.0   NaN   3.0  
CONFIG_rewardExists         None  None  None  None  
CONFIG_showNRounds           1.0   1.0   1.0   1.0  
CONFIG_showOtherSummaries    1.0   1.0   1.0   1.0  
CONFIG_showPunishmentId      NaN  None   NaN  None  
Controled_Or_Observational  True  True  True  True  
DV_efficiencyReported          1     1     1     1  
Empirical                   True  True  True  True  
Lab_Or_Field                True  True  True  True

## 7. DV_efficiencyReported — where do human and LLM disagree?

In [ ]:
eff_df = df_cmp[df_cmp['field'] == 'DV_efficiencyReported'].copy()
eff_df['human_norm'] = eff_df['human'].astype(str).str.strip().str.lower()
eff_df['llm_norm']   = eff_df['llm'].astype(str).str.strip().str.lower()
eff_df['match'] = eff_df['human_norm'] == eff_df['llm_norm']
display(eff_df[['paper', 'condition (human)', 'condition (LLM)', 'human', 'llm', 'match']])

,paper,condition (human),condition (LLM),human,llm,match
19,10.1007_s10640-025-00970-6,Condition 0,"NoR – Part NoP (No Representation, No Punishment)",0.0,1,False
39,10.1007_s10640-025-00970-6,Condition 1,"NoR – Part P (No Representation, Punishment)",0.0,1,False
59,10.1007_s10640-025-00970-6,Condition 2,"RnoM – Part NoP (Representation, No Messages, ...",0.0,1,False
79,10.1007_s10640-025-00970-6,Condition 3,"RnoM – Part P (Representation, No Messages, Pu...",0.0,1,False
99,10.1007_s10640-025-00970-6,Condition 4,"RM – Part NoP (Representation, Messages, No Pu...",0.0,1,False
119,10.1007_s10640-025-00970-6,Condition 5,"RM – Part P (Representation, Messages, Punishm...",0.0,1,False
139,10.1007_s10645-008-9094-1,Condition 0,"Treatment 1 (T1) – Homogeneous MPCR, No Punish...",1.0,1,False
159,10.1007_s10645-008-9094-1,Condition 1,"Treatment 2 (T2) – Heterogeneous MPCR, No Puni...",1.0,1,False
179,10.1007_s10645-008-9094-1,Condition 2,"Treatment 3 (T3) – Heterogeneous MPCR, Punishm...",1.0,1,False
199,10.1016_j.evolhumbehav.2006.06.001,Condition 0,Experiment 1 – Anonymous,0.0,0,False
